In [1]:
import logging
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import svm
from scipy import stats

In [2]:
# prompt: mount google drive in this ipynb

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# prompt: write code to unzip data and save it in drive

import zipfile
import os

# Define the path to the zip file in your Google Drive
zip_file_path = '/content/drive/MyDrive/FYP/data/data.zip'  # Replace with your actual path

# Define the directory where you want to extract the data
extract_path = '/content/drive/My Drive/FYP/data/unzipped' # Replace with your desired path

# Create the extract directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

try:
  with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
  print(f'Successfully unzipped {zip_file_path} to {extract_path}')
except FileNotFoundError:
  print(f'Error: Zip file not found at {zip_file_path}')
except Exception as e:
  print(f'An error occurred: {e}')


Successfully unzipped /content/drive/MyDrive/FYP/data/data.zip to /content/drive/My Drive/FYP/data/unzipped


In [4]:
# def get_train(*args):
#     x = np.load('/content/drive/MyDrive/FYP/data/unzipped/ali_train_x.npy').astype(np.float32)
#     y = np.load('/content/drive/MyDrive/FYP/data/unzipped/ali_train_y.npy').astype(np.float32)
#     # (184602, 16) (184602,)
#     # 184532 : 70
#     ind = [y==0]
#     x_train = x[ind]
#     y_train = y[ind]

#     scaler = MinMaxScaler()
#     scaler.fit(x_train)
#     scaler.transform(x_train)

#     return x_train, y_train

In [5]:
def get_train(*args):
    """Get training dataset for KDD 10 percent"""
    return _get_adapted_dataset("train")

In [6]:
def get_test(*args):
    """Get testing dataset for KDD 10 percent"""
    return _get_adapted_dataset("test")

In [7]:
def get_shape_input():
    """Get shape of the dataset for KDD 10 percent"""
    return (None, 121)


In [8]:
def get_shape_label():
    """Get shape of the labels in KDD 10 percent"""
    return (None,)

In [9]:
def _get_dataset():
    """ Gets the basic dataset
    Returns :
            dataset (dict): containing the data
                dataset['x_train'] (np.array): training images shape
                (?, 120)
                dataset['y_train'] (np.array): training labels shape
                (?,)
                dataset['x_test'] (np.array): testing images shape
                (?, 120)
                dataset['y_test'] (np.array): testing labels shape
                (?,)
    """
    col_names = _col_names()
    df = pd.read_csv("/content/drive/MyDrive/FYP/data/kddcup.data_10_percent_corrected", header=None, names=col_names)

    text_l = ['protocol_type', 'service', 'flag', 'land', 'logged_in', 'is_host_login', 'is_guest_login']

    for name in text_l:
        _encode_text_dummy(df, name)

    labels = df['label'].copy()
    # Given the ratio of normal(less) and abnormal samples(more), suppose preponderant samples are normal
    # Isolation Tree regards the dominant labels as 1(normal), and the less as -1(abnormal)
    labels[labels != 'normal.'] = 1
    labels[labels == 'normal.'] = -1

    df['label'] = labels

    df_train = df.sample(frac=0.5, random_state=42)
    df_test = df.loc[~df.index.isin(df_train.index)]

    x_train, y_train = _to_xy(df_train, target='label')
    y_train = y_train.flatten().astype(int)
    x_test, y_test = _to_xy(df_test, target='label')
    y_test = y_test.flatten().astype(int)

    # x_train = x_train[y_train != 1]
    # y_train = y_train[y_train != 1]

    scaler = MinMaxScaler()
    scaler.fit(x_train)
    scaler.transform(x_train)
    scaler.transform(x_test)

    dataset = {}
    dataset['x_train'] = x_train.astype(np.float32)
    dataset['y_train'] = y_train.astype(np.float32)
    dataset['x_test'] = x_test.astype(np.float32)
    dataset['y_test'] = y_test.astype(np.float32)

    return dataset

In [10]:
def _get_adapted_dataset(split):
    """ Gets the adapted dataset for the experiments

    Args :
            split (str): train or test
    Returns :
            (tuple): <training, testing> images and labels
    """
    dataset = _get_dataset()
    key_img = 'x_' + split
    key_lbl = 'y_' + split

    if split != 'train':
        dataset[key_img], dataset[key_lbl] = _adapt(dataset[key_img],
                                                    dataset[key_lbl])

    return (dataset[key_img], dataset[key_lbl])


In [11]:
def _encode_text_dummy(df, name):
    """Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1]
    for red,green,blue)
    """
    dummies = pd.get_dummies(df.loc[:,name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df.loc[:, dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)

In [12]:
def _to_xy(df, target):
    """Converts a Pandas dataframe to the x,y inputs that TensorFlow needs"""
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    dummies = df[target]
    return df.to_numpy().astype(np.float32), dummies.to_numpy().astype(np.float32)


In [13]:
def _col_names():
    """Column names of the dataframe 42-dim"""
    return ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"]


In [14]:
def _adapt(x, y, rho=0.1):
    """Adapt the ratio of normal/anomalous data"""

    # Normal data: label =0, anomalous data: label =1

    rng = np.random.RandomState(42) # seed shuffling

    inliersx = x[y == 1]
    inliersy = y[y == 1]
    outliersx = x[y == -1]
    outliersy = y[y == -1]

    size_outliers = outliersx.shape[0]
    inds = rng.permutation(size_outliers)
    outliersx, outliersy = outliersx[inds], outliersy[inds]

    size_test = inliersx.shape[0]
    out_size_test = int(size_test*rho/(1-rho))
    outestx = outliersx[:out_size_test]
    outesty = outliersy[:out_size_test]

    testx = np.concatenate((inliersx,outestx), axis=0)
    testy = np.concatenate((inliersy,outesty), axis=0)

    size_test = testx.shape[0]
    inds = rng.permutation(size_test)
    testx, testy = testx[inds], testy[inds]

    return testx, testy

In [15]:
trainx, trainy = get_train()
testx, testy = get_test()

In [24]:
# from sklearn.svm import OneClassSVM
# from sklearn.metrics import accuracy_score
# import numpy as np

# clf = OneClassSVM(nu=0.95 * 0.1 + 0.05, kernel="rbf", gamma=0.1)

# display_interval = 1000

# best_accuracy = 0
# no_improvement_count = 0
# max_no_improvement = 25  # Stop if no improvement for this many intervals

# for i in range(0, len(trainx), display_interval):
#     clf.fit(trainx[i : i + display_interval])
# #
#     y_pred = clf.predict(testx)

#     accuracy = accuracy_score(testy, y_pred)

#     print(f"Iteration: {i}, Accuracy: {accuracy}")

#     if accuracy > best_accuracy:
#         best_accuracy = accuracy
#         no_improvement_count = 0
#     else:
#         no_improvement_count += 1
#         if no_improvement_count >= max_no_improvement:
#             print("Stopping training: No improvement for", max_no_improvement, "intervals.")
#             break

Iteration: 0, Accuracy: 0.2109978949660654


KeyboardInterrupt: 

In [25]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score
import numpy as np
import joblib  # For saving and loading models

clf = OneClassSVM(nu=0.95 * 0.1 + 0.05, kernel="rbf", gamma=0.1)

display_interval = 1000

best_accuracy = 0
no_improvement_count = 0
max_no_improvement = 25  # Stop if no improvement for this many intervals
best_model_path = 'best_model.pkl'  # Path to save the best model


for i in range(0, len(trainx), display_interval):
    clf.fit(trainx[i: i + display_interval])

    y_pred = clf.predict(testx)

    accuracy = accuracy_score(testy, y_pred)

    print(f"Iteration: {i}, Accuracy: {accuracy}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        no_improvement_count = 0
        # Save the current model as the best model
        joblib.dump(clf, best_model_path)
        print(f"Saved best model with accuracy {best_accuracy} to {best_model_path}")
    else:
        no_improvement_count += 1
        if no_improvement_count >= max_no_improvement:
            print("Stopping training: No improvement for", max_no_improvement, "intervals.")
            break

# Load the best model for later use
best_clf = joblib.load(best_model_path)
print("Best model loaded from:", best_model_path)

Iteration: 0, Accuracy: 0.2109978949660654
Saved best model with accuracy 0.2109978949660654 to best_model.pkl
Iteration: 1000, Accuracy: 0.29685968860015244
Saved best model with accuracy 0.29685968860015244 to best_model.pkl
Iteration: 2000, Accuracy: 0.28219703843501615
Iteration: 3000, Accuracy: 0.6449025514463035
Saved best model with accuracy 0.6449025514463035 to best_model.pkl
Iteration: 4000, Accuracy: 0.6469213878706493
Saved best model with accuracy 0.6469213878706493 to best_model.pkl
Iteration: 5000, Accuracy: 0.2858717743984321
Iteration: 6000, Accuracy: 0.28291383878343557
Iteration: 7000, Accuracy: 0.20772238957645264
Iteration: 8000, Accuracy: 0.7277066018219431
Saved best model with accuracy 0.7277066018219431 to best_model.pkl
Iteration: 9000, Accuracy: 0.716206039269771
Iteration: 10000, Accuracy: 0.29323485645846187
Iteration: 11000, Accuracy: 0.7261595833484557
Iteration: 12000, Accuracy: 0.19357238776176822
Iteration: 13000, Accuracy: 0.6452201212209197
Iteration

In [29]:
clf = best_clf

In [26]:
y_pred = clf.predict(testx)

In [27]:
n_errors = (y_pred != testy).sum()

In [28]:
print(testy.shape,y_pred.shape)
print('testy -1:',(testy == -1).sum())
print('y_pred -1:',(y_pred == -1).sum())
posi_num = 0

for i in range(0, len(testy)):
    if testy[i] == y_pred[i] and testy[i] == -1:
        posi_num += 1
print('posi_num', posi_num)
n_errors = (y_pred != testy).sum()
print("Total errors:", n_errors, "Accuracy:", 1 - n_errors / testx.shape[0], 'Precision:', posi_num / (y_pred == -1).sum())

(220424,) (220424,)
testy -1: 22042
y_pred -1: 178027
posi_num 21715
Total errors: 156639 Accuracy: 0.28937411534134216 Precision: 0.12197588006313649
